[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_6_change_detection/assignments/change_detection.ipynb)

# Assignment 6B: UAS change detection

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

In this assignment we detect change at Lake Wheeler between two UAS surveys: July 17, 2024 and September 17, 2025. The 2024 survey is georeferenced; the 2025 survey was processed without ground control points. Its orthophoto is displaced by several meters, and its DSM is in an arbitrary vertical reference and is bent into a dome. Before any change can be measured, the 2025 products have to be co-registered to the 2024 ones. We then detect change in three ways: by differencing a spectral index, by differencing the DSMs with a level of detection, and by comparing two classified maps.

Keep in mind that a difference between two surveys contains real change, registration error, and measurement noise. The level of detection is the size of difference that the noise alone can produce; only differences larger than that are reported as change. The two flights are also from different months, so part of the spectral difference is the season and not a change in land cover.

This notebook runs the same steps as the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/assignment_6b.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed.

## Learning objectives

- Recognize a survey without ground control from its offsets against a reference survey
- Co-register an orthophoto horizontally and a DSM vertically using stable surfaces
- Normalize a spectral index between dates and map its change with a threshold
- Compute a DEM of Difference, estimate a level of detection, and report significant change and volumes
- Compare two classified maps with a from-to matrix

## Outline

- Part 1: Environment setup
- Part 2: Import the two surveys
- Part 3: Horizontal co-registration
- Part 4: Index differencing
- Part 5: DSM co-registration
- Part 6: Level of detection and significant change
- Part 7: Post-classification comparison
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba, together with scipy, which the level of detection tool uses. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5 scipy pandas matplotlib

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import numpy, pandas, and matplotlib.

In [ ]:
import io
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.script import array as garray
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB)
- The polygons of Assignment 6A, `classification_training.gpkg`; we use its road polygons as surfaces that did not change

The orthophotos and DSMs of the two flights are GeoTIFFs on the course bucket; `r.import` reads only the part of them we ask for, so they are not downloaded as files.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
FLIGHTS = f"{BUCKET}/uas-flight-data/Lake%20Wheeler%20-%20NCSU"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
TRAINING_URL = f"{BUCKET}/grassdata/classification_training.gpkg"
ORTHO_2024_URL = f"{FLIGHTS}/071724/odm_orthophoto.tif"
DSM_2024_URL = f"{FLIGHTS}/071724/dsm.tif"
ORTHO_2025_URL = f"{FLIGHTS}/091725/ortho_2025_09_17.cog.tif"
DSM_2025_URL = f"{FLIGHTS}/091725/dsm.tif"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!curl -sSL -o classification_training.gpkg {TRAINING_URL}
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `change_detection` inside the Lake Wheeler project, start the session there, and create a `Tools` object.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "change_detection"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session, overwrite=True)

print(tools.g_proj(flags="p").text)

### Install the addon

The [r.dem](https://grass.osgeo.org/grass85/manuals/addons/r.dem.html) toolset holds the tools for DEM differencing used in Parts 5 and 6: `r.dem.coregister`, `r.dem.dod`, `r.dem.bias`, `r.dem.lod`, and `r.dem.change`. Addons installed after the `Tools` object was created are called through the helper `run` below, which also returns the statistics that these tools print.

In [ ]:
gs.run_command("g.extension", extension="r.dem", quiet=True)
print("r.dem toolset installed")

### Helpers

`run` runs a GRASS tool by name and returns the text it prints. `describe` prints the statistics of a raster in the current region, optionally only under a mask raster. `show` draws a raster with an optional legend and vector overlay. `rgb` draws a three band composite. `full_region` and `farm_region` switch the computational region between the whole site at 0.5 m and a 150 m block around the farm buildings that we use for detail maps.

In [ ]:
N, S, W, E, RES = 219815, 219070, 636565, 637355, 0.5


def run(tool, **kwargs):
    """Run a GRASS tool and return its printed output (messages included)."""
    p = gs.start_command(tool, overwrite=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, **kwargs)
    out, err = p.communicate()
    text = out + err
    if isinstance(text, bytes):
        text = text.decode()
    if p.returncode != 0:
        raise RuntimeError(f"{tool} failed:\n{text}")
    return "\n".join(l for l in text.splitlines() if "%" not in l and l.strip())


def describe(raster, label=None, zones=None):
    """Print statistics of a raster in the current region; with zones, only where zones is not null."""
    name = raster
    if zones:
        tools.r_mapcalc(expression=f"tmp_describe = if(isnull({zones}), null(), {raster})")
        name = "tmp_describe"
    s = tools.r_univar(map=name, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells, min {s['min']:.2f}, max {s['max']:.2f}, mean {s['mean']:.3f}, median {s['median']:.3f}, stddev {s['stddev']:.3f}")
    return s


def show(raster, title=None, legend=None, width=600, height=565, flags="b", vector=None, at=(5, 45, 3, 6), base=None):
    """Draw one raster in the current region, optionally over an RGB base, with a legend."""
    m = gj.Map(width=width, height=height, use_region=True)
    if base:
        m.d_rgb(red=f"{base}.1", green=f"{base}.2", blue=f"{base}.3")
    m.d_rast(map=raster)
    if vector:
        m.d_vect(map=vector, color="black", fill_color="none", width=2)
    if legend is not None:
        m.d_legend(raster=raster, at=at, title=legend, flags=flags, bgcolor="white", fontsize=11)
    m.d_barscale(at=(55, 5), flags="n")
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()


def rgb(prefix, title=None, width=600, height=565, vector=None):
    """Draw a true color composite of the bands prefix.1, prefix.2, prefix.3."""
    m = gj.Map(width=width, height=height, use_region=True)
    m.d_rgb(red=f"{prefix}.1", green=f"{prefix}.2", blue=f"{prefix}.3")
    if vector:
        m.d_vect(map=vector, color="yellow", fill_color="none", width=2)
    m.d_barscale(at=(55, 5), flags="n")
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()


def full_region():
    """Set the computational region to the whole site at 0.5 m."""
    tools.g_region(n=N, s=S, w=W, e=E, res=RES)


def farm_region():
    """Set the computational region to the 150 m block around the farm buildings."""
    tools.g_region(n=219450, s=219300, w=637030, e=637180, res=0.1)

---
## Part 2: Import the two surveys

### The reference survey, July 17, 2024

We work at 0.5 m on the same extent as Assignment 6A. The 2024 orthophoto and DSM were processed in WebODM with georeferencing; they are the reference. `r.import` reads, reprojects, and resamples them to the region.

In [ ]:
full_region()
print(tools.g_region(flags="p").text)
tools.r_import(input=f"/vsicurl/{ORTHO_2024_URL}", output="ortho_2024", extent="region", resolution="region", resample="nearest")
tools.r_import(input=f"/vsicurl/{DSM_2024_URL}", output="dsm_2024", extent="region", resolution="region", resample="bilinear")
describe("dsm_2024", "DSM 2024");

### The survey without ground control, September 17, 2025

The 2025 products are already in the coordinate system of the project. We import a window that is 20 m larger than the region on every side, because we expect to move the data. These rasters keep their original resolution (7 cm for the orthophoto, 8.5 cm for the DSM), so the import takes two to three minutes.

In [ ]:
tools.g_region(n=N + 20, s=S - 20, w=W - 20, e=E + 20, res=RES)
tools.r_import(input=f"/vsicurl/{ORTHO_2025_URL}", output="ortho_2025_raw", extent="region", resolution="region", resample="nearest")
tools.r_import(input=f"/vsicurl/{DSM_2025_URL}", output="dsm_2025_raw", extent="region", resolution="region", resample="bilinear")
full_region()
describe("dsm_2025_raw", "DSM 2025 as delivered");

The 2024 DSM ranges from about 75 to 116 m, the elevation of the site above the vertical datum. The 2025 DSM ranges from about -14 to 64 m: it is not referenced to the datum, and its range is almost twice as large.

#### Map: the farm buildings in both orthophotos

The yellow outlines are the building, road, and car polygons of Assignment 6A, digitized on the 2024 orthophoto.

In [ ]:
tools.v_import(input=str(DATA_DIR / "classification_training.gpkg"), layer="training", output="training")
farm_region()
rgb("ortho_2024", title="July 17, 2024", width=450, height=450, vector="training")
rgb("ortho_2025_raw", title="September 17, 2025, as delivered", width=450, height=450, vector="training")
full_region()

> **Think about it:** In which direction, and by roughly how many meters, are the buildings displaced in the 2025 orthophoto?

---
## Part 3: Horizontal co-registration

### Measure the offset

To measure the displacement we use surfaces that look the same in both images: the gravel roads, the roofs, and the concrete pads, which are the brightest surfaces of the site. For a block of the site, `offset` makes a mask of the brightest 7 percent of cells in each image and finds the shift, in steps of one 0.5 m cell, at which the two masks agree best (the peak of their cross-correlation). We measure it in 200 m blocks every 100 m and keep the blocks where the masks correlate well, which are the blocks along the roads.

In [ ]:
def band_mean(prefix, n, s, w, e):
    """Mean of the three bands of an orthophoto as a numpy array for a block at 0.5 m."""
    env = os.environ.copy()
    env["GRASS_REGION"] = gs.region_env(n=n, s=s, w=w, e=e, res=RES)
    return sum(garray.array(f"{prefix}.{i}", null=0, env=env).astype(float) for i in (1, 2, 3)) / 3


def bright(image):
    """Mask of the brightest 7 percent of the valid cells."""
    valid = image[image > 0]
    if valid.size == 0:
        return np.zeros_like(image)
    return (image > np.percentile(valid, 93)).astype(float)


def offset(reference, moving, x, y, half=100, maxlag=50):
    """Shift (east, north, in m) to apply to `moving` to match `reference` in a block centered at x, y."""
    a = bright(band_mean(reference, y + half, y - half, x - half, x + half))
    b = bright(band_mean(moving, y + half, y - half, x - half, x + half))
    if a.sum() < 200 or b.sum() < 200:
        return None
    a, b = a - a.mean(), b - b.mean()
    cc = np.fft.fftshift(np.fft.ifft2(np.fft.fft2(a) * np.conj(np.fft.fft2(b))).real)
    cy, cx = np.array(cc.shape) // 2
    window = cc[cy - maxlag:cy + maxlag + 1, cx - maxlag:cx + maxlag + 1]
    row, col = np.unravel_index(np.argmax(window), window.shape)
    quality = window.max() / np.sqrt((a * a).sum() * (b * b).sum())
    return (col - maxlag) * RES, -(row - maxlag) * RES, quality


def offset_table(reference, moving, min_quality=0.4):
    rows = []
    for y in range(219150, 219800, 100):
        for x in range(636700, 637350, 100):
            result = offset(reference, moving, x, y)
            if result and result[2] >= min_quality:
                rows.append({"x": x, "y": y, "east (m)": result[0], "north (m)": result[1], "correlation": round(float(result[2]), 2)})
    return pd.DataFrame(rows)


offsets = offset_table("ortho_2024", "ortho_2025_raw")
offsets

The east offset is nearly the same in every block. The north offset is not: it is close to zero in the south and grows to about 11 m in the north. A single shift cannot remove it. We describe the offset with the simplest model that fits: a constant east offset, and a north offset that changes linearly with the northing. In a few blocks the correlation locks onto a different feature and returns an offset far from its neighbors, so the line is fitted twice: once to all blocks, and again without the blocks that lie more than 3 m from the first fit.

In [ ]:
Y0 = 219400
DX = float(offsets["east (m)"].median())
# Fit, drop blocks more than 3 m from the fit or from the median east offset, and fit again.
C1, C0 = np.polyfit(offsets["y"] - Y0, offsets["north (m)"], 1)
keep = ((offsets["north (m)"] - (C0 + C1 * (offsets["y"] - Y0))).abs() <= 3) & ((offsets["east (m)"] - DX).abs() <= 3)
used = offsets[keep]
C1, C0 = np.polyfit(used["y"] - Y0, used["north (m)"], 1)
residual = used["north (m)"] - (C0 + C1 * (used["y"] - Y0))
print(f"blocks used: {len(used)} of {len(offsets)}")
print(f"east offset: {DX:.1f} m (standard deviation {used['east (m)'].std():.1f} m)")
print(f"north offset: {C0:.2f} + {C1:.4f} * (northing - {Y0}) m, residual standard deviation {residual.std():.1f} m")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(offsets["y"], offsets["north (m)"], "o", label="north offset")
ax.plot(offsets["y"], offsets["east (m)"], "s", label="east offset")
ys = np.array([S, N])
ax.plot(ys, C0 + C1 * (ys - Y0), "-", color="C0")
ax.axhline(DX, color="C1")
ax.set_xlabel("Northing (m)")
ax.set_ylabel("Offset of 2025 against 2024 (m)")
ax.legend()
fig.tight_layout()
plt.show()

### Apply the correction

A raster in GRASS is a grid of cells plus the coordinates of its edges. `r.region` changes those coordinates without touching the cells. Moving all four edges by the same amount shifts the raster; moving the north and south edges by different amounts also stretches it. That is exactly the model above, so we copy the 2025 rasters and give the copies corrected edges: each east and west edge moves by the east offset, and each north and south edge moves by the north offset at its corrected northing.

The corrected rasters are then aggregated from their original resolution to the 0.5 m cells of the region with `r.resamp.stats` (the mean of the cells that fall into each 0.5 m cell; the minimum for the alpha band).

In [ ]:
def corrected_edges(raster):
    info = tools.r_info(map=raster, flags="g", format="json").json
    north = (info["north"] + C0 - C1 * Y0) / (1 - C1)
    south = (info["south"] + C0 - C1 * Y0) / (1 - C1)
    return dict(n=north, s=south, w=info["west"] + DX, e=info["east"] + DX)


for band in ("1", "2", "3", "4"):
    tools.g_copy(raster=[f"ortho_2025_raw.{band}", f"ortho_2025_moved.{band}"])
    tools.r_region(map=f"ortho_2025_moved.{band}", **corrected_edges(f"ortho_2025_raw.{band}"))
    tools.r_resamp_stats(input=f"ortho_2025_moved.{band}", output=f"ortho_2025.{band}",
                         method="minimum" if band == "4" else "average")
tools.g_copy(raster=["dsm_2025_raw", "dsm_2025_moved"])
tools.r_region(map="dsm_2025_moved", **corrected_edges("dsm_2025_raw"))
tools.r_resamp_stats(input="dsm_2025_moved", output="dsm_2025_h", method="average")
print(corrected_edges("dsm_2025_raw"))

### Check the result

Measure the offsets again, now between the 2024 orthophoto and the corrected 2025 orthophoto.

In [ ]:
check = offset_table("ortho_2024", "ortho_2025")
print(f"remaining offset: east median {check['east (m)'].median():.1f} m, north median {check['north (m)'].median():.1f} m, "
      f"largest {max(check['east (m)'].abs().max(), check['north (m)'].abs().max()):.1f} m")
check

#### Map: the farm buildings after the correction

In [ ]:
farm_region()
rgb("ortho_2024", title="July 17, 2024", width=450, height=450, vector="training")
rgb("ortho_2025", title="September 17, 2025, corrected", width=450, height=450, vector="training")
full_region()

The correction is a model with three numbers, fitted to blocks along the roads on the east side of the site. Offsets of 1 to 3 m remain. Everything that follows compares the two dates cell by cell at 0.5 m, so edges of objects will show apparent change that is only this remaining misregistration.

We restrict the analysis to the cells covered by both surveys with a mask.

In [ ]:
tools.r_mapcalc(expression="both = if(ortho_2024.4 > 0 && ortho_2025.4 > 0 && !isnull(dsm_2024) && !isnull(dsm_2025_h), 1, null())")
tools.r_mask(raster="both")
area = tools.r_univar(map="both", format="json").json["n"] * RES * RES / 10000
print(f"area covered by both surveys: {area:.1f} ha")

#### Map: the whole site at both dates

In [ ]:
rgb("ortho_2024", title="July 17, 2024")
rgb("ortho_2025", title="September 17, 2025")

---
## Part 4: Index differencing

### VARI at both dates

We compute VARI for both dates with the expression of Assignment 6A.

In [ ]:
for year in (2024, 2025):
    r, g, b = (f"ortho_{year}.{i}" for i in (1, 2, 3))
    tools.r_mapcalc(expression=f"vari_{year} = if({g} + {r} - {b} <= 25, null(), max(-1.0, min(1.0, float({g} - {r}) / ({g} + {r} - {b}))))")
    tools.r_colors(map=f"vari_{year}", color="ndvi")
    describe(f"vari_{year}")

### Relative normalization

The two orthophotos come from different cameras, processing software, and light. A surface that did not change should have the same index value at both dates, so we use the gravel roads as pseudo-invariant features: the road polygons of Assignment 6A. The difference between the median VARI of the road cells at the two dates is the radiometric offset between the images, and we subtract it from the 2025 index.

In [ ]:
tools.v_extract(input="training", where="class_id = 5", output="roads")
tools.v_to_rast(input="roads", output="roads_rast", use="val", value=1)
road_2024 = describe("vari_2024", "VARI 2024 on roads", zones="roads_rast")["median"]
road_2025 = describe("vari_2025", "VARI 2025 on roads", zones="roads_rast")["median"]
radiometric_offset = road_2025 - road_2024
print(f"radiometric offset (2025 minus 2024 on roads): {radiometric_offset:.3f}")
tools.r_mapcalc(expression=f"vari_2025n = vari_2025 - ({radiometric_offset})")
tools.r_colors(map="vari_2025n", color="ndvi");

### The difference and a threshold

The difference map is the normalized 2025 index minus the 2024 index: negative where vegetation was lost, positive where it was gained.

In [ ]:
tools.r_mapcalc(expression="vari_diff = vari_2025n - vari_2024")
tools.r_colors(map="vari_diff", rules=io.StringIO("-1 139:69:19\n-0.1 222:184:135\n0 white\n0.1 144:238:144\n1 0:100:0\n"))
stats = describe("vari_diff")
road_noise = describe("vari_diff", "VARI difference on roads", zones="roads_rast")

#### Map and histogram: VARI at both dates and the difference

In [ ]:
show("vari_2024", title="VARI, July 2024", legend="VARI")
show("vari_2025n", title="VARI, September 2025, normalized", legend="VARI")
show("vari_diff", title="VARI difference, 2025 minus 2024", legend="difference")

difference = garray.array("vari_diff", null=np.nan)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(difference[np.isfinite(difference)], bins=100, range=(-0.6, 0.6), color="#4c8c4a")
for x in (-0.1, 0.1):
    ax.axvline(x, color="black", linestyle="--")
ax.set_xlabel("VARI difference, 2025 minus 2024")
ax.set_ylabel("Cells")
fig.tight_layout()
plt.show()

A threshold turns the difference into a change map. We take 0.1: the standard deviation of the difference on the road cells, which did not change, is well below it. Cells below -0.1 are a decrease of vegetation, cells above 0.1 an increase.

In [ ]:
THRESHOLD = 0.1
tools.r_mapcalc(expression=f"vari_change = if(vari_diff < -{THRESHOLD}, 1, if(vari_diff > {THRESHOLD}, 3, 2))")
tools.r_category(map="vari_change", separator="pipe", rules=io.StringIO("1|vegetation decrease\n2|no change\n3|vegetation increase\n"))
tools.r_colors(map="vari_change", rules=io.StringIO("1 139:69:19\n2 230:230:230\n3 0:128:0\n"))


def areas(raster):
    """Area in hectares and percent of each category of a raster in the current region."""
    text = tools.r_stats(input=raster, flags="anl", separator="|").text
    table = pd.read_csv(io.StringIO(text), sep="|", names=["category", "label", "area_m2"])
    table["hectares"] = (table["area_m2"] / 10000).round(2)
    table["percent"] = (100 * table["area_m2"] / table["area_m2"].sum()).round(1)
    return table[["category", "label", "hectares", "percent"]]


show("vari_change", title="VARI change, threshold 0.1", legend="", flags="cb", at=(5, 25, 3, 6))
areas("vari_change")

> **Think about it:** July is the middle of the growing season and mid September is harvest time. Which part of the decrease is a change of land cover, and which part would reverse by next July?

---
## Part 5: DSM co-registration

The 2025 DSM now lines up horizontally (`dsm_2025_h`), but its elevations are not in the vertical datum. We bring it to the 2024 surface in two steps with the `r.dem` tools: a vertical shift measured on the roads, and then a correction of the dome.

### Vertical shift from pseudo ground control points

`r.dem.coregister` samples the difference between the two DSMs on features that are assumed not to have changed, here the road polygons, and subtracts the median difference. Its report gives the bias and the NMAD (normalized median absolute deviation), a robust standard deviation of the remaining difference on those features.

In [ ]:
print(run("r.dem.coregister", dem="dsm_2025_h", reference="dsm_2024", pgcp="roads", output="dsm_2025_v",
          method="pgcp_vertical", buffer=0.5))

The bias is about -90 m, and after removing it the roads still disagree by several meters. `r.dem.dod` computes the DEM of Difference (DoD), the later surface minus the earlier one.

In [ ]:
print(run("r.dem.dod", dem="dsm_2025_v", reference="dsm_2024", output="dod_shifted", range=5))
describe("dod_shifted", "DoD after the vertical shift")
describe("dod_shifted", "DoD after the vertical shift, on roads", zones="roads_rast")
show("dod_shifted", title="DoD after the vertical shift", legend="m")

The DoD is not noise around zero. It is a smooth shape several meters high: one sign in the middle of the site and the other toward the edges. This is the dome, a known distortion of surfaces reconstructed from nadir images without ground control: an error in the estimated lens distortion bends the whole model. A shift cannot remove it, and neither can a rotation.

### Stable surfaces

To model the dome we need many cells, spread over the whole site, where the real surface did not change. The roads alone are too few and all on one side. Bare ground is a larger candidate: a cell with VARI below 0 at both dates carried no green vegetation at either date, so its surface is soil, gravel, or a roof. We also require a slope of less than 8 degrees within one meter in the 2024 DSM, to exclude edges of buildings and trees, where the remaining horizontal offset turns into large height differences.

For an independent check we divide the stable cells by a 100 m checkerboard: two thirds of the blocks are used to fit the correction, one third is held out.

In [ ]:
tools.r_slope_aspect(elevation="dsm_2024", slope="slope_2024")
tools.r_neighbors(input="slope_2024", output="slope_2024_max", size=5, method="maximum")
tools.r_mapcalc(expression="stable = if(vari_2024 < 0 && vari_2025 < 0 && slope_2024_max < 8, 1, null())")
tools.r_mapcalc(expression="holdout_block = (int(x() / 100) + int(y() / 100)) % 3 == 0")
tools.r_mapcalc(expression="stable_fit = if(stable && !holdout_block, 1, null())")
tools.r_mapcalc(expression="stable_check = if(stable && holdout_block, 1, null())")
tools.r_colors(map="stable_fit", rules=io.StringIO("1 yellow\n"))
tools.r_colors(map="stable_check", rules=io.StringIO("1 cyan\n"))
for name in ("stable", "stable_fit", "stable_check"):
    n = tools.r_univar(map=name, format="json").json["n"]
    print(f"{name}: {n:,} cells, {n * RES * RES / 10000:.1f} ha")
m = gj.Map(width=600, height=565, use_region=True)
m.d_rgb(red="ortho_2024.1", green="ortho_2024.2", blue="ortho_2024.3")
m.d_rast(map="stable_fit")
m.d_rast(map="stable_check")
m.d_barscale(at=(55, 5), flags="n")
m.d_text(text="Stable cells: fit (yellow), held out (cyan)", at=(3, 94), size=4, color="black", bgcolor="white")
m.show()

### Remove the dome

`r.dem.bias` with `method=spline` interpolates the DoD of the stable fit cells into a smooth surface, the bias field, and subtracts it from the DoD. The spline is fitted at 10 m resolution with strong smoothing, because the dome is a long wavelength shape.

In [ ]:
print(run("r.dem.bias", dod="dod_shifted", output="dod", method="spline", stable_mask="stable_fit", bias_field="dome"))
tools.r_colors(map="dome", color="differences")
describe("dome", "bias field")
fit = describe("dod", "corrected DoD on the fit cells", zones="stable_fit")
check = describe("dod", "corrected DoD on the held out cells", zones="stable_check")
roads_after = describe("dod", "corrected DoD on the road polygons", zones="roads_rast")

#### Map: the bias field and the corrected DoD

In [ ]:
show("dome", title="Bias field (the dome)", legend="m")
tools.r_colors(map="dod", rules=io.StringIO("-30 0:0:139\n-2 blue\n-0.3 white\n0.3 white\n2 red\n30 139:0:0\n"))
show("dod", title="Corrected DoD, 2025 minus 2024", legend="m", flags="b")

> **Think about it:** Compare the standard deviation of the corrected DoD on the fit cells and on the held out cells. Why is the second the one to report?

---
## Part 6: Level of detection and significant change

### Level of detection

The level of detection (LoD) is the smallest difference that can be told apart from noise at a chosen confidence. `r.dem.lod` estimates the noise as the NMAD of the corrected DoD on stable cells and multiplies it by the critical value of the normal distribution, 1.96 for 95 percent. We give it the held out cells, which the correction did not see.

In [ ]:
print(run("r.dem.lod", dod="dod", output="lod", method="global", stable_mask="stable_check", confidence=0.95))
LOD = tools.r_univar(map="lod", format="json").json["max"]
print(f"LoD at 95 percent: {LOD:.2f} m")

### Significant change and volumes

`r.dem.change` keeps the cells of the DoD whose absolute value exceeds the LoD, removes isolated cells with the `-n` flag, and sums the volumes of surface raising and surface lowering. The tool uses the terms of geomorphology, deposition and erosion; on this site a raised surface is mostly taller vegetation and a lowered surface mostly vegetation that was removed.

In [ ]:
print(run("r.dem.change", dod="dod", lod="lod", output_sig="dod_significant",
          volume_csv=str(DATA_DIR / "volume.csv"), flags="n"))
tools.r_colors(map="dod_significant", rules=io.StringIO("-30 0:0:139\n-2 blue\n0 white\n2 red\n30 139:0:0\n"))
pd.read_csv(DATA_DIR / "volume.csv")

In [ ]:
tools.r_mapcalc(expression=f"dod_class = if(isnull(dod_significant), 2, if(dod_significant < 0, 1, 3))")
tools.r_category(map="dod_class", separator="pipe", rules=io.StringIO("1|surface lowered\n2|below the level of detection\n3|surface raised\n"))
tools.r_colors(map="dod_class", rules=io.StringIO("1 blue\n2 230:230:230\n3 red\n"))
show("dod_significant", title="Significant change over the 2025 orthophoto", legend="m", base="ortho_2025")
areas("dod_class")

#### Map: significant change around the farm buildings

In [ ]:
farm_region()
show("dod_significant", title="Significant change", legend="m", width=450, height=450, base="ortho_2025")
full_region()

### What changed in the fields

The mean of the corrected DoD inside the cells that lost vegetation according to Part 4, and inside the cells with no VARI change, connects the two results.

In [ ]:
for category, label in [(1, "vegetation decrease"), (2, "no change"), (3, "vegetation increase")]:
    tools.r_mapcalc(expression=f"tmp_zone = if(vari_change == {category}, 1, null())")
    describe("dod", f"DoD where VARI shows {label}", zones="tmp_zone")

> **Think about it:** Which of the significant changes are real changes of the surface (crop, trees, buildings, vehicles), and which follow outlines of objects that did not change? What produces the second kind?

---
## Part 7: Post-classification comparison

The third way to detect change is to classify each date on its own and compare the two maps. We use a simple classification that can be applied to both orthophotos: a cell is vegetation if its VARI is above 0.05 and bare or built otherwise, with the normalized index for 2025. `r.stats` on the two maps together gives the from-to matrix: the area of each combination of class in 2024 and class in 2025.

In [ ]:
for year, index in ((2024, "vari_2024"), (2025, "vari_2025n")):
    tools.r_mapcalc(expression=f"class_{year} = if({index} > 0.05, 1, 2)")
    tools.r_category(map=f"class_{year}", separator="pipe", rules=io.StringIO("1|vegetation\n2|bare or built\n"))
    tools.r_colors(map=f"class_{year}", rules=io.StringIO("1 0:128:0\n2 222:184:135\n"))
text = tools.r_stats(input=["class_2024", "class_2025"], flags="anl", separator="|").text
pairs = pd.read_csv(io.StringIO(text), sep="|", names=["id_2024", "class 2024", "id_2025", "class 2025", "area_m2"])
matrix = (pairs.pivot(index="class 2024", columns="class 2025", values="area_m2") / 10000).round(2)
matrix["total 2024"] = matrix.sum(axis=1)
matrix.loc["total 2025"] = matrix.sum()
matrix

#### Map: the two classified maps and the from-to map

In [ ]:
show("class_2024", title="Classes, July 2024", legend="", flags="cb", at=(5, 20, 3, 6))
show("class_2025", title="Classes, September 2025", legend="", flags="cb", at=(5, 20, 3, 6))
tools.r_mapcalc(expression="from_to = (class_2024 - 1) * 2 + class_2025")
tools.r_category(map="from_to", separator="pipe", rules=io.StringIO(
    "1|vegetation, unchanged\n2|vegetation to bare\n3|bare to vegetation\n4|bare, unchanged\n"))
tools.r_colors(map="from_to", rules=io.StringIO("1 0:128:0\n2 139:69:19\n3 144:238:144\n4 230:230:230\n"))
show("from_to", title="From-to classes", legend="", flags="cb", at=(5, 30, 3, 6))
areas("from_to")

### The three methods side by side

The area each method reports as a loss of vegetation: VARI decrease (Part 4), surface lowered beyond the LoD (Part 6), and vegetation to bare (this part), and how much of each coincides with the others.

In [ ]:
tools.r_mapcalc(expression="agree = if(vari_change == 1, 1, 0) + if(dod_class == 1, 2, 0) + if(from_to == 2, 4, 0)")
text = tools.r_stats(input="agree", flags="an", separator="|").text
combo = {int(l.split("|")[0]): float(l.split("|")[1]) / 10000 for l in text.splitlines()}
total = lambda bit: sum(a for k, a in combo.items() if k & bit)
print(f"VARI decrease:            {total(1):6.2f} ha")
print(f"surface lowered:          {total(2):6.2f} ha")
print(f"vegetation to bare:       {total(4):6.2f} ha")
print(f"all three agree:          {combo.get(7, 0):6.2f} ha")
print(f"VARI decrease and lowered:{combo.get(3, 0) + combo.get(7, 0):6.2f} ha")
tools.r_mask(flags="r");

> **Think about it:** The three methods answer different questions. Which one would you report for "how much crop was harvested", and which for "where did the land cover change"?

---
## Report questions

Answer in the cells below, or in your report document, as described on the [assignment page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/assignment_6b.html). Keep the answers short and refer to the outputs above.

#### Question 1: The uncontrolled survey

Report the elevation ranges of the two DSMs and the offset table of the orthophotos. What do they tell you about how the 2025 survey was processed?

`Add text here with your answer to question 1`

#### Question 2: Horizontal co-registration

Report the offset model, the remaining offsets after the correction, and the farm building maps before and after. Where is the correction least reliable, and why?

`Add text here with your answer to question 2`

#### Question 3: Index differencing

Report the radiometric offset, the VARI difference map, and the areas of the three change classes. How much of the decrease do you attribute to the season?

`Add text here with your answer to question 3`

#### Question 4: DSM co-registration

Report the vertical bias, the DoD after the vertical shift, the bias field, and the standard deviation of the corrected DoD on the fit cells, the held out cells, and the roads. What is the dome, and why did the vertical shift not remove it?

`Add text here with your answer to question 4`

#### Question 5: Level of detection and change

Report the LoD, the map of significant change, and the volumes. Which changes are real, and which are artifacts of the registration?

`Add text here with your answer to question 5`

#### Question 6: Comparison of methods

Report the from-to matrix and the areas of vegetation loss from the three methods. Why do they differ, and what would ground control points in the 2025 survey have changed in this assignment?

`Add text here with your answer to question 6`